<a href="https://colab.research.google.com/github/sanchita-88/Alpha-Council/blob/main/IMDB_Review_Sentiment_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1 — Install dependencies
!pip install -q pandas numpy scikit-learn matplotlib seaborn nltk tensorflow torch transformers datasets accelerate

In [ ]:
# Cell 2 — Import libraries and set seeds
import pandas as pd
import numpy as np
import random
import re
import warnings
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# NLP and preprocessing
import nltk
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Deep Learning - TensorFlow/Keras
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Bidirectional, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam

# Transformers
import torch
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

# Download NLTK data
nltk.download('stopwords', quiet=True)

# Set seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("All libraries imported successfully!")
print(f"TensorFlow version: {tf.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")

In [ ]:
# Cell 3 — Load and inspect dataset
# Load the dataset
df = pd.read_csv('/content/IMDB Dataset.csv')

print("Dataset loaded successfully!")
print(f"\nDataset shape: {df.shape}")
print(f"\nFirst few rows:")
print(df.head())

print(f"\nDataset info:")
print(df.info())

print(f"\nClass distribution:")
print(df['sentiment'].value_counts())

# Visualize class distribution
plt.figure(figsize=(8, 5))
df['sentiment'].value_counts().plot(kind='bar', color=['green', 'red'])
plt.title('Class Distribution in IMDb Dataset')
plt.xlabel('Sentiment')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.show()

# Convert labels: positive → 1, negative → 0
df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})

print(f"\nLabel encoding:")
print(df[['sentiment', 'label']].drop_duplicates())
print(f"\nLabel distribution:")
print(df['label'].value_counts())

In [ ]:
# Cell 4 — Train/test split (stratified)
X = df['review'].values
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")
print(f"\nTraining set class distribution:")
print(pd.Series(y_train).value_counts())
print(f"\nTest set class distribution:")
print(pd.Series(y_test).value_counts())

In [ ]:
# Cell 5 — Text preprocessing function
def preprocess_text(text):
    """
    Preprocesses text by:
    - Lowercasing
    - Removing HTML tags
    - Removing punctuation
    - Removing stopwords
    Returns cleaned text
    """
    # Lowercase
    text = text.lower()

    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)

    # Remove punctuation and special characters (keep only letters and spaces)
    text = re.sub(r'[^a-z\s]', '', text)

    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    words = text.split()
    words = [word for word in words if word not in stop_words]

    return ' '.join(words)

# Apply preprocessing
print("Preprocessing training data...")
X_train_clean = [preprocess_text(text) for text in X_train]

print("Preprocessing test data...")
X_test_clean = [preprocess_text(text) for text in X_test]

print(f"\nExample original review:\n{X_train[0][:300]}")
print(f"\nExample preprocessed review:\n{X_train_clean[0][:300]}")

In [ ]:
# Cell 6 — TF-IDF vectorization
print("Creating TF-IDF vectorizer...")
tfidf = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    min_df=2
)

print("Fitting on training data...")
X_train_tfidf = tfidf.fit_transform(X_train_clean)

print("Transforming test data...")
X_test_tfidf = tfidf.transform(X_test_clean)

print(f"\nTF-IDF matrix shape (train): {X_train_tfidf.shape}")
print(f"TF-IDF matrix shape (test): {X_test_tfidf.shape}")
print(f"Vocabulary size: {len(tfidf.vocabulary_)}")

In [ ]:
# Cell 7 — Logistic Regression with hyperparameter tuning
print("Training Logistic Regression with Grid Search...")

# Define parameter grid
param_grid_lr = {
    'C': [0.01, 0.1, 1, 5, 10]
}

# Create model
lr_model = LogisticRegression(
    solver='liblinear',
    random_state=SEED,
    max_iter=1000
)

# Grid search
grid_lr = GridSearchCV(
    lr_model,
    param_grid_lr,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

# Fit model
grid_lr.fit(X_train_tfidf, y_train)

print(f"\nBest parameters: {grid_lr.best_params_}")
print(f"Best cross-validation score: {grid_lr.best_score_:.4f}")

# Predict on test set
y_pred_lr = grid_lr.predict(X_test_tfidf)

# Calculate metrics
lr_accuracy = accuracy_score(y_test, y_pred_lr)
lr_precision = precision_score(y_test, y_pred_lr)
lr_recall = recall_score(y_test, y_pred_lr)
lr_f1 = f1_score(y_test, y_pred_lr)

print("\n" + "="*50)
print("LOGISTIC REGRESSION RESULTS")
print("="*50)
print(f"Accuracy:  {lr_accuracy:.4f}")
print(f"Precision: {lr_precision:.4f}")
print(f"Recall:    {lr_recall:.4f}")
print(f"F1 Score:  {lr_f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr, target_names=['Negative', 'Positive']))

# Confusion matrix
cm_lr = confusion_matrix(y_test, y_pred_lr)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'])
plt.title('Logistic Regression - Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

In [ ]:
# Cell 8 — Linear SVM with hyperparameter tuning
print("Training Linear SVM with Grid Search...")

# Define parameter grid
param_grid_svm = {
    'C': [0.01, 0.1, 1, 5, 10]
}

# Create model
svm_model = LinearSVC(
    random_state=SEED,
    max_iter=2000,
    dual=False
)

# Grid search
grid_svm = GridSearchCV(
    svm_model,
    param_grid_svm,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

# Fit model
grid_svm.fit(X_train_tfidf, y_train)

print(f"\nBest parameters: {grid_svm.best_params_}")
print(f"Best cross-validation score: {grid_svm.best_score_:.4f}")

# Predict on test set
y_pred_svm = grid_svm.predict(X_test_tfidf)

# Calculate metrics
svm_accuracy = accuracy_score(y_test, y_pred_svm)
svm_precision = precision_score(y_test, y_pred_svm)
svm_recall = recall_score(y_test, y_pred_svm)
svm_f1 = f1_score(y_test, y_pred_svm)

print("\n" + "="*50)
print("LINEAR SVM RESULTS")
print("="*50)
print(f"Accuracy:  {svm_accuracy:.4f}")
print(f"Precision: {svm_precision:.4f}")
print(f"Recall:    {svm_recall:.4f}")
print(f"F1 Score:  {svm_f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_svm, target_names=['Negative', 'Positive']))

# Confusion matrix
cm_svm = confusion_matrix(y_test, y_pred_svm)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_svm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'])
plt.title('Linear SVM - Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

In [ ]:
# Cell 9 — Tokenization and padding for BiLSTM
VOCAB_SIZE = 30000
MAX_LENGTH = 300

print("Creating tokenizer for BiLSTM...")
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

# Convert texts to sequences
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

# Pad sequences
X_train_padded = pad_sequences(X_train_seq, maxlen=MAX_LENGTH, padding='post', truncating='post')
X_test_padded = pad_sequences(X_test_seq, maxlen=MAX_LENGTH, padding='post', truncating='post')

print(f"\nVocabulary size: {len(tokenizer.word_index)}")
print(f"Training sequences shape: {X_train_padded.shape}")
print(f"Test sequences shape: {X_test_padded.shape}")

# Example
print(f"\nExample original text:\n{X_train[0][:200]}")
print(f"\nExample tokenized sequence (first 50 tokens):\n{X_train_seq[0][:50]}")
print(f"\nExample padded sequence shape: {X_train_padded[0].shape}")

In [ ]:
# Cell 10 — Download and load GloVe embeddings
print("Downloading GloVe embeddings...")
!wget -q http://nlp.stanford.edu/data/glove.6B.zip
!unzip -q glove.6B.zip

print("Loading GloVe embeddings...")
EMBEDDING_DIM = 100

embeddings_index = {}
with open('glove.6B.100d.txt', encoding='utf-8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype='float32')
        embeddings_index[word] = coefs

print(f"Loaded {len(embeddings_index)} word vectors")

# Create embedding matrix
vocab_size = min(len(tokenizer.word_index) + 1, VOCAB_SIZE)
embedding_matrix = np.zeros((vocab_size, EMBEDDING_DIM))

for word, i in tokenizer.word_index.items():
    if i >= VOCAB_SIZE:
        continue
    embedding_vector = embeddings_index.get(word)
    if embedding_vector is not None:
        embedding_matrix[i] = embedding_vector

print(f"\nEmbedding matrix shape: {embedding_matrix.shape}")
print(f"Number of words with GloVe vectors: {np.sum(np.any(embedding_matrix != 0, axis=1))}")

In [ ]:
# Cell 11 — Build BiLSTM model with GloVe embeddings
print("Building BiLSTM model...")

model_bilstm = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=EMBEDDING_DIM,
        weights=[embedding_matrix],
        input_length=MAX_LENGTH,
        trainable=False
    ),
    Bidirectional(LSTM(128, return_sequences=True)),
    Dropout(0.3),
    Bidirectional(LSTM(64)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

# Compile model
model_bilstm.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("\nModel architecture:")
model_bilstm.summary()

In [ ]:
# Cell 12 — Train BiLSTM model
print("Training BiLSTM model...")

# Define callbacks
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    ),
    ModelCheckpoint(
        'best_bilstm_model.h5',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

# Train model
history = model_bilstm.fit(
    X_train_padded, y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
    callbacks=callbacks,
    verbose=1
)

# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Accuracy
axes[0].plot(history.history['accuracy'], label='Train Accuracy')
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy')
axes[0].set_title('BiLSTM Model Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

# Loss
axes[1].plot(history.history['loss'], label='Train Loss')
axes[1].plot(history.history['val_loss'], label='Val Loss')
axes[1].set_title('BiLSTM Model Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

# Evaluate on test set
print("\nEvaluating BiLSTM on test set...")
y_pred_bilstm_proba = model_bilstm.predict(X_test_padded)
y_pred_bilstm = (y_pred_bilstm_proba > 0.5).astype(int).flatten()

# Calculate metrics
bilstm_accuracy = accuracy_score(y_test, y_pred_bilstm)
bilstm_precision = precision_score(y_test, y_pred_bilstm)
bilstm_recall = recall_score(y_test, y_pred_bilstm)
bilstm_f1 = f1_score(y_test, y_pred_bilstm)

print("\n" + "="*50)
print("BiLSTM WITH GLOVE RESULTS")
print("="*50)
print(f"Accuracy:  {bilstm_accuracy:.4f}")
print(f"Precision: {bilstm_precision:.4f}")
print(f"Recall:    {bilstm_recall:.4f}")
print(f"F1 Score:  {bilstm_f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_bilstm, target_names=['Negative', 'Positive']))

# Confusion matrix
cm_bilstm = confusion_matrix(y_test, y_pred_bilstm)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_bilstm, annot=True, fmt='d', cmap='Purples',
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'])
plt.title('BiLSTM - Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

In [ ]:
# Import transformer libraries
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import torch

torch.manual_seed(42)

print("Loading DistilBERT tokenizer...")
bert_tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
print("Tokenizer loaded!")

In [ ]:
# Prepare datasets
print("Preparing data for DistilBERT...")

train_df = pd.DataFrame({'text': X_train, 'label': y_train})
test_df = pd.DataFrame({'text': X_test, 'label': y_test})

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

In [ ]:
# Tokenize datasets with optimal max length
def tokenize_function(examples):
    return bert_tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=256  # Full length for better accuracy
    )

print("Tokenizing datasets...")
train_dataset = train_dataset.map(tokenize_function, batched=True, batch_size=1000)
test_dataset = test_dataset.map(tokenize_function, batched=True, batch_size=1000)

train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

print("Tokenization complete!")

In [ ]:
# Load DistilBERT model
print("Loading DistilBERT model...")
bert_model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2
)
print("Model loaded!")
print(f"Model parameters: {bert_model.num_parameters():,}")

In [ ]:
# Define metrics computation
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    return {
        'accuracy': accuracy_score(labels, predictions),
        'precision': precision_score(labels, predictions),
        'recall': recall_score(labels, predictions),
        'f1': f1_score(labels, predictions)
    }

print("Metrics function defined.")

In [ ]:
# Configure training arguments for optimal accuracy
training_args = TrainingArguments(
    output_dir='./distilbert_results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=500,
    logging_dir='./logs',
    logging_steps=500,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    greater_is_better=True,
    seed=42,
    fp16=False,
    report_to='none',
    save_total_limit=2
)

print("="*60)
print("Training Configuration:")
print(f"  - Epochs: 3")
print(f"  - Batch size: 16")
print(f"  - Learning rate: 2e-5")
print(f"  - Max length: 256")
print(f"  - Weight decay: 0.01")
print(f"  - Load best model: True")
print("="*60)

In [ ]:
# Create Trainer
trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

print("Trainer created!")

In [ ]:
# Fine-tune DistilBERT
print("="*60)
print("Starting DistilBERT fine-tuning...")
print("Expected accuracy: 0.92-0.94")
print("="*60)

trainer.train()

print("\n✓ Training complete!")

In [ ]:
# Evaluate on test set
print("\n" + "="*60)
print("EVALUATING DISTILBERT ON TEST SET")
print("="*60)

results = trainer.evaluate()

# Store results in required variables
bert_accuracy = results['eval_accuracy']
bert_precision = results['eval_precision']
bert_recall = results['eval_recall']
bert_f1 = results['eval_f1']

print("\n" + "="*60)
print("DISTILBERT FINAL RESULTS")
print("="*60)
print(f"Accuracy:  {bert_accuracy:.4f}")
print(f"Precision: {bert_precision:.4f}")
print(f"Recall:    {bert_recall:.4f}")
print(f"F1 Score:  {bert_f1:.4f}")
print("="*60)

# Check if target achieved
if bert_accuracy >= 0.91:
    print(f"\n✓✓✓ SUCCESS! Accuracy {bert_accuracy:.4f} EXCEEDS target (0.91)")
    print("     Research objective achieved!")
else:
    print(f"\n⚠ Accuracy {bert_accuracy:.4f} - target is 0.91")

improvement = bert_accuracy - 0.889
print(f"\nImprovement over previous run: {improvement:+.4f} ({improvement*100:+.2f}%)")

In [ ]:
# Get predictions for detailed analysis
print("\nGenerating predictions for analysis...")
predictions = trainer.predict(test_dataset)
y_pred_bert = np.argmax(predictions.predictions, axis=1)

print("\n" + "="*60)
print("DETAILED CLASSIFICATION REPORT")
print("="*60)
print(classification_report(y_test, y_pred_bert, target_names=['Negative', 'Positive'], digits=4))

In [ ]:
# Confusion matrix visualization
cm_bert = confusion_matrix(y_test, y_pred_bert)

plt.figure(figsize=(10, 8))
sns.heatmap(cm_bert, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'],
            cbar_kws={'label': 'Count'},
            annot_kws={'size': 14, 'weight': 'bold'})
plt.title(f'DistilBERT - Confusion Matrix\nAccuracy: {bert_accuracy:.4f}',
          fontsize=16, fontweight='bold', pad=20)
plt.ylabel('True Label', fontsize=13, fontweight='bold')
plt.xlabel('Predicted Label', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nConfusion Matrix Analysis:")
print(f"  True Negatives:  {cm_bert[0, 0]:,}")
print(f"  False Positives: {cm_bert[0, 1]:,}")
print(f"  False Negatives: {cm_bert[1, 0]:,}")
print(f"  True Positives:  {cm_bert[1, 1]:,}")

# Calculate error rate
total_errors = cm_bert[0, 1] + cm_bert[1, 0]
total_samples = cm_bert.sum()
error_rate = total_errors / total_samples
print(f"\n  Total errors: {total_errors:,} / {total_samples:,}")
print(f"  Error rate: {error_rate:.4f} ({error_rate*100:.2f}%)")

In [ ]:
# Summary comparison
print("\n" + "="*60)
print("PERFORMANCE COMPARISON")
print("="*60)
print(f"Previous DistilBERT result: 0.8890")
print(f"Current DistilBERT result:  {bert_accuracy:.4f}")
print(f"Improvement:                {bert_accuracy - 0.889:+.4f}")
print("="*60)
print(f"\nBaseline paper (BiLSTM):    0.9100")
print(f"Current DistilBERT:         {bert_accuracy:.4f}")
if bert_accuracy > 0.91:
    print(f"Status:                     ✓ EXCEEDS baseline by {(bert_accuracy - 0.91)*100:.2f}%")
else:
    print(f"Gap to baseline:            {0.91 - bert_accuracy:.4f}")
print("="*60)

print(f"\n✓ Results stored in variables:")
print(f"  bert_accuracy  = {bert_accuracy:.4f}")
print(f"  bert_precision = {bert_precision:.4f}")
print(f"  bert_recall    = {bert_recall:.4f}")
print(f"  bert_f1        = {bert_f1:.4f}")